In [ ]:
from docplex.mp.model import Model

# Parameters
T = 360  # Horizon length
N = 1   # Number of caissons

C_COPER = 15600
C_MANT = 48000
C_LEAK = 100000
K2 = 1.0
M = 3.6
P_threshold = 3.2
P_max = 3.5
b_COPER = 4
b_MANT = 50
mu_COPER = 1.6
mu_MANT = 4.2

P_init = {
    0: 3.5
}
alpha_init = {
    0: 0.00167
}

# Create the model
mdl = Model(name='full_optimized_model')

# Define decision variables
x = {(t, i): mdl.binary_var(name=f'x_{t}_{i}') for t in range(T) for i in range(N)}
y = {(t, i): mdl.binary_var(name=f'y_{t}_{i}') for t in range(T) for i in range(N)}
v = {(t, i): mdl.continuous_var(lb=0, name=f'v_{t}_{i}') for t in range(T) for i in range(N)}
P = {(t, i): mdl.continuous_var(name=f'P_{t}_{i}') for t in range(T) for i in range(N)}
alpha = {(t, i): mdl.continuous_var(name=f'alpha_{t}_{i}') for t in range(T) for i in range(N)}
f = {(t, i): mdl.continuous_var(name=f'f_{t}_{i}') for t in range(T) for i in range(N)}
g = {(t, i): mdl.continuous_var(name=f'g_{t}_{i}') for t in range(T) for i in range(N)}
o = {(t, i): mdl.binary_var(name=f'o_{t}_{i}') for t in range(T) for i in range(N)}

# Objective function
mdl.minimize(mdl.sum(C_COPER * x[t, i] + C_MANT * y[t, i] + K2 * C_LEAK * v[t, i] for t in range(T) for i in range(N)))

# Initial pressure constraint
for i in range(N):
    mdl.add_constraint(P[0, i] == P_init[i], ctname=f'initial_P_{i}')
    mdl.add_constraint(alpha[0, i] == alpha_init[i], ctname=f'initial_alpha_{i}')

# Pressure bounds
for t in range(T):
    for i in range(N):
        mdl.add_constraint(P[t, i] >= P_threshold, ctname=f'P_threshold_{t}_{i}')
        mdl.add_constraint(P[t, i] <= P_max, ctname=f'P_max_{t}_{i}')

# Pressure evolution constraints
for t in range(T-1):
    for i in range(N):
        mdl.add_constraint(P[t+1, i] >= (1 - alpha[t, i]) * P[t, i] - M * (f[t, i] + g[t, i]), ctname=f'evolution1_{t}_{i}')
        mdl.add_constraint(P[t+1, i] <= (1 - alpha[t, i]) * P[t, i] + M * (f[t, i] + g[t, i]), ctname=f'evolution2_{t}_{i}')

# Pressure reset after repair (COPER)
for t in range(T):
    for i in range(N):
        if (t - b_COPER) in range(T):
            mdl.add_constraint(P[t, i] >= P_max - M * (1 - x[t - b_COPER, i]), ctname=f'afterCOPER1_{t}_{i}')
            mdl.add_constraint(P[t, i] <= P_max + M * (1 - x[t - b_COPER, i]), ctname=f'afterCOPER2_{t}_{i}')

# Pressure reset after maintenance (MANT)
for t in range(T):
    for i in range(N):
        if (t - b_MANT) in range(T):
            mdl.add_constraint(P[t, i] >= P_max - M * (1 - y[t - b_MANT, i]), ctname=f'afterMANT1_{t}_{i}')
            mdl.add_constraint(P[t, i] <= P_max + M * (1 - y[t - b_MANT, i]), ctname=f'afterMANT2_{t}_{i}')

# Alpha reset after repair (COPER)
for t in range(T):
    for i in range(N):
        if (t - b_COPER) in range(T):
            mdl.add_constraint(alpha[t, i] >= (1/mu_COPER) * alpha[t-b_COPER, i] - M*(1 - x[t - b_COPER, i]), ctname=f'afterCOPER_alpha1_{t}_{i}')
            mdl.add_constraint(alpha[t, i] <= (1/mu_COPER) * alpha[t-b_COPER, i] + M*(1 - x[t - b_COPER, i]), ctname=f'afterCOPER_alpha2_{t}_{i}')

# Alpha reset after maintenance (MANT)
for t in range(T):
    for i in range(N):
        if (t - b_MANT) in range(T):
            mdl.add_constraint(alpha[t, i] >= (1/mu_MANT) * alpha[t-b_MANT, i] - M*(1 - y[t - b_MANT, i]), ctname=f'afterMANT_alpha1_{t}_{i}')
            mdl.add_constraint(alpha[t, i] <= (1/mu_MANT) * alpha[t-b_MANT, i] + M*(1 - y[t - b_MANT, i]), ctname=f'afterMANT_alpha2_{t}_{i}')

# State control variables f constraints (COPER)
for t in range(T):
    for i in range(N):
        # f[t+k,i] >= x[t,i] 对所有k
        for k in range(b_COPER):
            if t + k in range(T):
                mdl.add_constraint(f[t + k, i] >= x[t, i], ctname=f'def_f_{t+k}_{i}')
        # f[t,i] <= sum of x[tp,i] 
        start_tp = max(0, t - b_COPER)
        wsum = mdl.sum(x[tp, i] for tp in range(start_tp, t+1))
        mdl.add_constraint(f[t, i] <= wsum, ctname=f'rangef_{t}_{i}')

# State control variables g constraints (MANT)
for t in range(T):
    for i in range(N):
        # g[t+k,i] >= y[t,i] 对所有k
        for k in range(b_MANT):
            if t + k in range(T):
                mdl.add_constraint(g[t + k, i] >= y[t, i], ctname=f'def_g_{t+k}_{i}')
        # g[t,i] <= sum of y[tp,i]
        start_tp = max(0, t - b_MANT)
        wsum = mdl.sum(y[tp, i] for tp in range(start_tp, t+1))
        mdl.add_constraint(g[t, i] <= wsum, ctname=f'rangeg_{t}_{i}')

# Unique repair/maintenance constraints
for i in range(N):
    for t in range(T):
        for tp in range(T):
            if tp >= t + b_COPER:
                mdl.add_constraint(x[t, i] + x[tp, i] <= 1, ctname=f'xunique_{t}_{tp}_{i}')
            if tp >= t + b_MANT:
                mdl.add_constraint(y[t, i] + y[tp, i] <= 1, ctname=f'yunique_{t}_{tp}_{i}')

#  Absolute pressure change linearization
for t in range(1, T):
    for i in range(N):
            mdl.add_constraint(M * o[t, i] >= P[t-1, i] - P[t, i], ctname=f'defv1_{t}_{i}')
            mdl.add_constraint(M * (1 - o[t, i]) >= -(P[t-1, i] - P[t, i]), ctname=f'defv2_{t}_{i}')
            mdl.add_constraint(v[t, i] >= 0, ctname=f'defv3_{t}_{i}')
            mdl.add_constraint(v[t, i] >= (P[t-1, i] - P[t, i]), ctname=f'defv4_{t}_{i}')
            mdl.add_constraint(v[t, i] <= M * o[t, i], ctname=f'defv5_{t}_{i}')
            mdl.add_constraint(v[t, i] <= ((P[t-1, i] - P[t, i]) + M*(1 - o[t, i])), ctname=f'defv6_{t}_{i}')


# solve the model
solution = mdl.solve(log_output=True)

if solution:
    print("Optimal objective value:", solution.objective_value)
    # 根据需要打印决策变量值
else:
    print("No solution found")


Version identifier: 22.1.1.0 | 2022-11-27 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
CPLEX Error  5002: 'evolution1_1_0' is not convex.
Presolve time = 0.09 sec. (5.37 ticks)

Root node processing (before b&c):
  Real time             =    0.09 sec. (5.40 ticks)
Parallel b&c, 8 threads:
  Real time             =    0.00 sec. (0.00 ticks)
  Sync time (average)   =    0.00 sec.
  Wait time (average)   =    0.00 sec.
                          ------------
Total (root+branch&cut) =    0.09 sec. (5.40 ticks)
Error: Model has non-convex quadratic constraint, name='evolution1_1_0'
No solution found


In [ ]:
import matplotlib.pyplot as plt

# 假设你已经得到解的变量值，例如：
# P_values = {(t, i): P[t, i].solution_value for t in T for i in N}
# x_values = {(t, i): x[t, i].solution_value for t in T for i in N}
# y_values = {(t, i): y[t, i].solution_value for t in T for i in N}

# 为简便起见，以下代码示范单个设备（i=0）压力随时间变化
device_id = 0

# 提取压力数据
pressure_over_time = [P[t, device_id].solution_value for t in T]

# 提取COPER和MANT的维修状态（平均或总和，视需求）
C_oper_usage = [sum(x[t, i].solution_value for i in N) for t in T]
MANT_usage = [sum(y[t, i].solution_value for i in N) for t in T]

# 画压力变化
plt.figure(figsize=(10, 6))
plt.plot(T, pressure_over_time, label='压力', marker='o')

# 叠加显示维修的影响，可以用不同颜色或样式标记维修时间点
for t in T:
    if C_oper_usage[t] > 0.5:
        plt.scatter(t, pressure_over_time[t], color='red', label='COPER维 修改' if t == 0 else "")
    if MANT_usage[t] > 0.5:
        plt.scatter(t, pressure_over_time[t], color='blue', label='MANT维修' if t == 0 else "")

# 设置图例、标题、标签
plt.xlabel('时间')
plt.ylabel('压力')
plt.title('压力随时间变化及维修影响')
plt.legend()
plt.grid(True)
plt.show()
